# Synthetic MDP：预处理长度 $T_0$ 与 ratio 通信轮数 $T_0'$ 的敏感性实验

这个 notebook 在 Chen 等（2021）第 5.1 节的 **10-agent 人造环境**中研究

\[
D_M(T_0,T_0')
=\max_{1\le n\le N}\left\|M^n_{T_0,T_0'}A-I\right\|_{\mathrm{op}}.
\]

- $A$：联合目标策略下的精确总体矩阵，只用于事后评价；
- $\widehat A^n$：节点 $n$ 用 $T_0$ 个 Markov 预样本和 $T_0'$ 轮 local-ratio 通信估计的矩阵；
- $M^n=(\widehat A^n)^{-1}$。

实验同时比较全连接网络和环形网络。每次独立实验先生成一条最长轨迹，不同 $T_0$ 使用同一轨迹的前缀，从而避免把不同轨迹的随机差异误认为 $T_0$ 的影响。

In [ ]:
from __future__ import annotations

import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import numpy as np

OUTPUT_DIR = Path("preconditioner_sensitivity_result")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("NumPy:", np.__version__)
print("结果目录:", OUTPUT_DIR.resolve())

## 1. Synthetic MDP 环境与通信网络

- 10 个智能体，共享 10 个状态；
- 每个智能体有 2 个局部动作，因此共有 $2^{10}=1024$ 个联合动作；
- 行为策略为均匀策略；
- 目标策略是在行为策略上加入标准差 0.05 的高斯扰动后重新归一化；
- 转移概率、节点奖励从 $[0,1]$ 均匀分布生成；
- 状态特征为 5 维标准高斯向量并单位归一化；
- 折扣因子 $\gamma=0.95$。

全连接网络满足 $W_{nn}=0.8, W_{nm}=1/45$；环形网络满足 $W_{nn}=0.8$，左右邻居权重各为 0.1。

In [ ]:
class SyntheticMDP:
    """Chen et al. (2021), Section 5.1 synthetic 10-agent MDP."""

    states = 10
    actions = 2
    agents = 10
    start_state = 0

    def __init__(self, seed: int = 20_210_501, target_noise_std: float = .05):
        rng = np.random.default_rng(seed)
        self.seed = seed
        self.joint_action_count = self.actions ** self.agents
        indices = np.arange(self.joint_action_count, dtype=np.int32)
        bit_positions = np.arange(self.agents, dtype=np.int32)
        self.joint_action_bits = ((indices[:, None] >> bit_positions) & 1).astype(np.int8)
        self.joint_action_weights = (1 << bit_positions).astype(np.int32)

        transition_raw = rng.uniform(
            0.0, 1.0, size=(self.states, self.joint_action_count, self.states)
        )
        self.transition = transition_raw / transition_raw.sum(axis=2, keepdims=True)

        # 奖励在本实验中不进入 A，但仍按主实验的顺序生成，以保持同一个随机环境。
        self.reward = rng.uniform(
            0.0, 1.0,
            size=(self.states, self.joint_action_count, self.states, self.agents),
        )
        self.behavior = np.full(
            (self.agents, self.states, self.actions), 1.0 / self.actions
        )
        perturbed = self.behavior + rng.normal(
            0.0, target_noise_std, size=self.behavior.shape
        )
        perturbed = np.maximum(perturbed, 1e-6)
        self.target = perturbed / perturbed.sum(axis=2, keepdims=True)

        features = rng.normal(size=(self.states, 5))
        self.features = features / np.linalg.norm(features, axis=1, keepdims=True)

    def behavior_transition_matrix(self) -> np.ndarray:
        # 均匀且相互独立的行为策略使每个联合动作概率均为 2^{-10}。
        return self.transition.mean(axis=1)

    def sample_pilot_trajectory(
        self, rng: np.random.Generator, length: int, burn_in: int
    ) -> dict[str, np.ndarray]:
        state = self.start_state
        for _ in range(burn_in):
            action = rng.integers(self.actions, size=self.agents, dtype=np.int8)
            joint_action = int(action @ self.joint_action_weights)
            state = int(rng.choice(self.states, p=self.transition[state, joint_action]))

        states = np.empty(length + 1, dtype=np.int16)
        local_ratio = np.empty((length, self.agents), dtype=float)
        states[0] = state
        agent_index = np.arange(self.agents)
        for t in range(length):
            action = rng.integers(self.actions, size=self.agents, dtype=np.int8)
            joint_action = int(action @ self.joint_action_weights)
            next_state = int(
                rng.choice(self.states, p=self.transition[state, joint_action])
            )
            local_ratio[t] = (
                self.target[agent_index, state, action]
                / self.behavior[agent_index, state, action]
            )
            states[t + 1] = next_state
            state = next_state
        return {"states": states, "local_rho": local_ratio}


def stationary_distribution(transition: np.ndarray) -> np.ndarray:
    """求行为策略 Markov 链的平稳分布。"""
    size = transition.shape[0]
    system = transition.T - np.eye(size)
    rhs = np.zeros(size)
    system[-1] = 1.0
    rhs[-1] = 1.0
    stationary = np.linalg.solve(system, rhs)
    stationary = np.maximum(stationary, 0.0)
    return stationary / stationary.sum()


def synthetic_network_matrix(network: str, agents: int = 10) -> np.ndarray:
    """构造 Synthetic MDP 使用的全连接或环形双随机矩阵。"""
    if network == "fully_connected":
        matrix = np.full((agents, agents), 1.0 / 45.0)
        np.fill_diagonal(matrix, .8)
    elif network == "ring":
        matrix = np.zeros((agents, agents))
        for n in range(agents):
            matrix[n, n] = .8
            matrix[n, (n - 1) % agents] = .1
            matrix[n, (n + 1) % agents] = .1
    else:
        raise ValueError(f"未知网络：{network}")
    if not np.allclose(matrix.sum(axis=0), 1.0) or not np.allclose(
        matrix.sum(axis=1), 1.0
    ):
        raise RuntimeError("W 不是双随机矩阵")
    return matrix


def communicated_ratios(
    local_ratio: np.ndarray, mixing: np.ndarray, rounds: int
) -> np.ndarray:
    """Chen 的 log-consensus：exp(N W^L log(local ratios))。"""
    mixed_log = np.log(local_ratio) @ np.linalg.matrix_power(mixing, rounds).T
    return np.exp(mixing.shape[0] * mixed_log)

## 2. 精确矩阵 $A$ 与有限通信的总体误差下限

精确矩阵为

\[
A=\mathbb E_{s\sim\mu_b,\,a\sim\pi}
\left[\phi(s)\bigl(\phi(s)-\gamma\phi(s')\bigr)^\top\right].
\]

当 $T_0\to\infty$ 但 $T_0'$ 有限时，每个节点仍可能使用带通信偏差的 ratio。因此实验同时计算总体误差下限，区分“样本不够”和“通信不够”两种误差来源。

In [ ]:
def exact_joint_A(
    env: SyntheticMDP, stationary: np.ndarray, gamma: float
) -> np.ndarray:
    """联合目标策略下的精确 A。"""
    dim = env.features.shape[1]
    result = np.zeros((dim, dim))
    agent_index = np.arange(env.agents)
    for state in range(env.states):
        phi = env.features[state]
        target_action_probability = env.target[
            agent_index[:, None], state, env.joint_action_bits.T
        ].prod(axis=0)
        expected_next_feature = env.transition[state] @ env.features
        outer = np.einsum(
            "i,aj->aij",
            phi,
            phi[None, :] - gamma * expected_next_feature,
        )
        result += stationary[state] * np.einsum(
            "a,aij->ij", target_action_probability, outer, optimize=True
        )
    return result


def population_residual_floors(
    env: SyntheticMDP,
    stationary: np.ndarray,
    gamma: float,
    mixing: np.ndarray,
    rounds_values: np.ndarray,
    exact_a: np.ndarray,
) -> np.ndarray:
    """计算 T0→∞ 时，仅由有限 T0' 通信造成的残差下限。"""
    dim = env.features.shape[1]
    identity = np.eye(dim)
    agent_index = np.arange(env.agents)
    floors = np.empty(len(rounds_values))
    for ell, rounds in enumerate(rounds_values):
        finite_a = np.zeros((env.agents, dim, dim))
        for state in range(env.states):
            phi = env.features[state]
            expected_next_feature = env.transition[state] @ env.features
            outer = np.einsum(
                "i,aj->aij",
                phi,
                phi[None, :] - gamma * expected_next_feature,
            )
            local_ratio = (
                env.target[agent_index[:, None], state, env.joint_action_bits.T].T
                / .5
            )
            ratios = communicated_ratios(local_ratio, mixing, int(rounds))
            finite_a += stationary[state] * np.einsum(
                "an,aij->nij", ratios, outer, optimize=True
            ) / env.joint_action_count
        inverses = np.linalg.inv(finite_a)
        floors[ell] = max(
            float(np.linalg.norm(inverse @ exact_a - identity, 2))
            for inverse in inverses
        )
    return floors

## 3. 最常修改的实验参数

下面的范围经过试跑，能够同时包含快速下降区、转折区和平台区：

- $T_0$ 从 50 增加到 12800，并额外保留主收敛实验使用的 $T_0=999$；
- $T_0'$ 从 2 增加到 120，满足当前 Chen 实验中 $T_0'>1$ 的要求；
- 默认 20 条独立 Markov 轨迹，图中画中位数，不画分位数阴影。

In [ ]:
# ==================== 调参主要在这里 ====================
RUNS = 20
T0_VALUES = np.array(
    [50, 100, 200, 400, 800, 999, 1_600, 3_200, 6_400, 12_800],
    dtype=int,
)
T0_PRIME_VALUES = np.array([2, 3, 5, 8, 12, 20, 40, 80, 120], dtype=int)
NETWORKS = ("fully_connected", "ring")

GAMMA = .95
BURN_IN = 1_000
ENVIRONMENT_SEED = 20_210_501
TRAJECTORY_SEED = 91_000
CONDITION_LIMIT = 1e12

# T0 曲线只画这些 T0'，避免九条线挤在一起。
T0_CURVE_ROUNDS = (2, 5, 12, 40, 120)
# T0' 曲线只画这些 T0，包含主实验的 T0=999。
T0_PRIME_CURVE_SIZES = (200, 999, 3_200, 12_800)

print("T0:", T0_VALUES)
print("T0':", T0_PRIME_VALUES)
print("重复次数:", RUNS)

## 4. 用同一条最长轨迹的前缀计算整个敏感性网格

对于每条轨迹、每个网络和每个 $T_0'$，程序逐段累加

\[
\widehat A^n=
\frac1{T_0}\sum_{t=0}^{T_0-1}
\widehat\rho_t^n(T_0')
\phi_t(\phi_t-\gamma\phi_{t+1})^\top.
\]

小 $T_0$ 与大 $T_0$ 使用相同轨迹的不同长度前缀，因此曲线更加平稳，也更适合观察真实的“肘部”。

In [ ]:
def preconditioner_residual_grid(
    env: SyntheticMDP,
    trajectory: dict[str, np.ndarray],
    mixing: np.ndarray,
    exact_a: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """返回一个 T0' × T0 的残差网格和条件数网格。"""
    phi = env.features[trajectory["states"][:-1]]
    difference = phi - GAMMA * env.features[trajectory["states"][1:]]
    outer = np.einsum("ti,tj->tij", phi, difference)
    dim = env.features.shape[1]
    identity = np.eye(dim)
    residual = np.full((len(T0_PRIME_VALUES), len(T0_VALUES)), np.nan)
    conditions = np.full_like(residual, np.nan)

    for ell, rounds in enumerate(T0_PRIME_VALUES):
        ratios = communicated_ratios(
            trajectory["local_rho"], mixing, int(rounds)
        )
        running_sum = np.zeros((env.agents, dim, dim))
        previous = 0
        for p, size in enumerate(T0_VALUES):
            # 只累加新增加的样本段，避免每个 T0 都从头计算。
            running_sum += np.einsum(
                "tn,tij->nij",
                ratios[previous:size],
                outer[previous:size],
                optimize=True,
            )
            estimate = running_sum / float(size)
            cond = np.linalg.cond(estimate)
            conditions[ell, p] = float(np.max(cond))
            if np.all(np.isfinite(cond)) and np.max(cond) <= CONDITION_LIMIT:
                inverses = np.linalg.inv(estimate)
                residual[ell, p] = max(
                    float(np.linalg.norm(inverse @ exact_a - identity, 2))
                    for inverse in inverses
                )
            previous = int(size)
    return residual, conditions


def elbow_index(x: np.ndarray, y: np.ndarray) -> int:
    """在 log(x)-线性(y) 归一化后，找距离端点连线最远的下降转折点。"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(y)
    if valid.sum() < 3 or np.nanmax(y) <= np.nanmin(y):
        return int(np.flatnonzero(valid)[0])
    x_normalized = (
        np.log(x) - np.log(x).min()
    ) / (np.log(x).max() - np.log(x).min())
    y_normalized = (y - np.nanmin(y)) / (np.nanmax(y) - np.nanmin(y))
    score = (1.0 - x_normalized) - y_normalized
    score[~valid] = -np.inf
    return int(np.argmax(score))

In [ ]:
# 创建与主收敛实验完全相同的随机环境，并精确计算 A。
env = SyntheticMDP(seed=ENVIRONMENT_SEED)
stationary = stationary_distribution(env.behavior_transition_matrix())
exact_a = exact_joint_A(env, stationary, GAMMA)

# 两种网络复用同一批轨迹，确保网络差别不是随机样本造成的。
pilot_trajectories = [
    env.sample_pilot_trajectory(
        np.random.default_rng(TRAJECTORY_SEED + run),
        int(T0_VALUES[-1]),
        BURN_IN,
    )
    for run in range(RUNS)
]

print("A condition number:", np.linalg.cond(exact_a))
print("已生成", len(pilot_trajectories), "条共享预采样轨迹")

In [ ]:
# residuals 的形状：network × run × T0' × T0。
residuals = np.empty((len(NETWORKS), RUNS, len(T0_PRIME_VALUES), len(T0_VALUES)))
conditions = np.empty_like(residuals)
population_floors = np.empty((len(NETWORKS), len(T0_PRIME_VALUES)))

for network_index, network in enumerate(NETWORKS):
    mixing = synthetic_network_matrix(network, env.agents)
    population_floors[network_index] = population_residual_floors(
        env, stationary, GAMMA, mixing, T0_PRIME_VALUES, exact_a
    )
    for run, trajectory in enumerate(pilot_trajectories):
        residuals[network_index, run], conditions[network_index, run] = (
            preconditioner_residual_grid(env, trajectory, mixing, exact_a)
        )
        print(f"{network}: run {run + 1}/{RUNS}", flush=True)

if np.any(~np.isfinite(residuals)):
    print("警告：部分小 T0 组合由于病态矩阵没有有限结果。")
else:
    print("全部组合均得到有限结果。")

## 5. 保存原始数据与统计结果

主图使用 20 次独立运行的中位数。CSV 同时保存均值、10%/90% 分位数、满足 $D_M\le 1/2$ 的比例，以及 $T_0\to\infty$ 时的总体误差下限。

In [ ]:
median_residual = np.nanmedian(residuals, axis=1)
q10_residual, q90_residual = np.nanquantile(residuals, [.10, .90], axis=1)
pass_rate = np.nanmean(residuals <= 1.0, axis=1)

np.savez_compressed(
    OUTPUT_DIR / "raw_preconditioner_sensitivity.npz",
    networks=np.asarray(NETWORKS),
    T0_values=T0_VALUES,
    T0_prime_values=T0_PRIME_VALUES,
    residuals=residuals,
    conditions=conditions,
    population_floors=population_floors,
)

with (OUTPUT_DIR / "summary_preconditioner_sensitivity.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.writer(handle)
    writer.writerow((
        "network", "T0", "T0_prime", "median", "mean", "q10", "q90",
        "pass_rate_le_half", "population_floor",
    ))
    for network_index, network in enumerate(NETWORKS):
        for ell, rounds in enumerate(T0_PRIME_VALUES):
            for p, size in enumerate(T0_VALUES):
                values = residuals[network_index, :, ell, p]
                writer.writerow((
                    network, int(size), int(rounds),
                    float(np.nanmedian(values)), float(np.nanmean(values)),
                    float(np.nanquantile(values, .10)),
                    float(np.nanquantile(values, .90)),
                    float(np.nanmean(values <= .5)),
                    float(population_floors[network_index, ell]),
                ))

print("原始数据与 CSV 已保存。")

## 6. 固定 $T_0'$，观察增加 $T_0$ 的作用

每条实线对应一个固定的通信轮数。相同颜色的虚线是该通信轮数对应的总体误差下限：当曲线靠近虚线后，继续增加样本的收益会迅速减小，形成自然的平台和“肘部”。

In [ ]:

# Publication export: preserve the numerical curves and use embedded vector fonts.
plt.rcParams.update({"font.family": "DejaVu Sans", "mathtext.fontset": "stix",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7})
NETWORK_LABELS = {
    "fully_connected": "Fully connected network",
    "ring": "Ring network",
}
COLORS = plt.cm.viridis(np.linspace(.08, .92, len(T0_CURVE_ROUNDS)))


def draw_residual_vs_T0(network_index: int, network: str) -> Path:
    """图 A：固定多个 T0'，横轴改变 T0。"""
    fig, axis = plt.subplots(figsize=(3.2, 2.5), constrained_layout=True)
    for color, rounds in zip(COLORS, T0_CURVE_ROUNDS):
        ell = int(np.flatnonzero(T0_PRIME_VALUES == rounds)[0])
        axis.plot(
            T0_VALUES,
            median_residual[network_index, ell],
            marker="o",
            markersize=2.4,
            linewidth=1.1,
            color=color,
            label=rf"$T_0'={rounds}$",
        )
        #axis.axhline(
        #    population_floors[network_index, ell],
        #    color=color,
        #    linestyle=":",
        #    linewidth=1.0,
        #    alpha=.72,
        #)

    # 在通信最充分的曲线上标出几何意义下的下降转折点。
    strongest_ell = int(np.flatnonzero(T0_PRIME_VALUES == T0_CURVE_ROUNDS[-1])[0])
    strongest_curve = median_residual[network_index, strongest_ell]
    knee = elbow_index(T0_VALUES, strongest_curve)
    #axis.scatter(
    #    [T0_VALUES[knee]], [strongest_curve[knee]],
    #    marker="*", s=150, color="#d45b1f", zorder=5, label="elbow",
    #)
    #axis.annotate(
    #    rf"$T_0\approx{T0_VALUES[knee]:,}$",
    #    xy=(T0_VALUES[knee], strongest_curve[knee]),
    #    xytext=(12, 18), textcoords="offset points",
    #    arrowprops={"arrowstyle": "->", "color": "#d45b1f"},
    #    fontsize=7,
    #)

    axis.axhline(1.0, color="#555555", linestyle="--", linewidth=1.2,
                 label=r"$D_M=1$")
    axis.set_xscale("log")
    visible_ticks = [int(v) for v in T0_VALUES if int(v) != 999]
    axis.set_xticks(visible_ticks)
    axis.set_xticklabels([f"{value:,}" for value in visible_ticks], rotation=40, ha="right")
    axis.set_xlim(T0_VALUES[0] / 1.18, T0_VALUES[-1] * 1.12)
    visible = [
        median_residual[network_index, np.flatnonzero(T0_PRIME_VALUES == r)[0]]
        for r in T0_CURVE_ROUNDS
    ]
    axis.set_ylim(0.0, 1.08 * np.nanmax(visible))
    axis.set_xlabel(r"$T_0$", fontsize=8)
    axis.set_ylabel(r"$\max_n\|M^nA-I\|_{\mathrm{op}}$", fontsize=8)
    axis.set_title(NETWORK_LABELS[network], fontsize=8)
    axis.grid(True, alpha=.25)
    axis.spines[["top", "right"]].set_visible(False)
    axis.legend(frameon=False, ncol=2, fontsize=6.5, loc="upper right")
    path = OUTPUT_DIR / f"residual_vs_T0_{network}.png"
    fig.savefig(path, dpi=800, bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return path


T0_FIGURES = [
    draw_residual_vs_T0(network_index, network)
    for network_index, network in enumerate(NETWORKS)
]
print("T0 曲线:", *T0_FIGURES, sep="\n")

## 7. 固定 $T_0$，观察增加 $T_0'$ 的作用

每条实线对应一个固定的预采样长度。黑色点划线是 $T_0\to\infty$ 时的通信误差下限。全连接网络应该更早进入平台；环形网络需要更多轮通信。

In [ ]:

# Publication export: preserve the numerical curves and use embedded vector fonts.
plt.rcParams.update({"font.family": "DejaVu Sans", "mathtext.fontset": "stix",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7})
COLORS_T0_PRIME = plt.cm.plasma(np.linspace(.08, .88, len(T0_PRIME_CURVE_SIZES)))


def draw_residual_vs_T0_prime(network_index: int, network: str) -> Path:
    """图 B：固定多个 T0，横轴改变 T0'。"""
    fig, axis = plt.subplots(figsize=(3.2, 2.5), constrained_layout=True)
    for color, size in zip(COLORS_T0_PRIME, T0_PRIME_CURVE_SIZES):
        p = int(np.flatnonzero(T0_VALUES == size)[0])
        axis.plot(
            T0_PRIME_VALUES,
            median_residual[network_index, :, p],
            marker="o",
            markersize=2.4,
            linewidth=1.1,
            color=color,
            label=rf"$T_0={size:,}$",
        )

    #axis.plot(
    #   T0_PRIME_VALUES,
    #    population_floors[network_index],
    #    color="#333333",
    #    linestyle="-.",
    #    linewidth=1.8,
    #    marker="s",
    #    markersize=4,
    #    label=r"$T_0\to\infty$ floor",
    #)

    # 在最长预采样曲线上标出通信轮数的下降转折点。
    largest_p = int(np.flatnonzero(T0_VALUES == T0_PRIME_CURVE_SIZES[-1])[0])
    largest_curve = median_residual[network_index, :, largest_p]
    knee = elbow_index(T0_PRIME_VALUES, largest_curve)
    #axis.scatter(
    #    [T0_PRIME_VALUES[knee]], [largest_curve[knee]],
    #    marker="*", s=150, color="#087e8b", zorder=5, label="elbow",
    #)
    #axis.annotate(
    #    rf"$T_0'\approx{T0_PRIME_VALUES[knee]}$",
    #    xy=(T0_PRIME_VALUES[knee], largest_curve[knee]),
    #    xytext=(12, 18), textcoords="offset points",
    #    arrowprops={"arrowstyle": "->", "color": "#087e8b"},
    #    fontsize=7,
    #)

    axis.axhline(1.0, color="#555555", linestyle="--", linewidth=1.2,
                 label=r"$D_M=1$")
    axis.set_xscale("log")
    axis.set_xticks(T0_PRIME_VALUES)
    axis.set_xticklabels([str(value) for value in T0_PRIME_VALUES])
    axis.set_xlim(T0_PRIME_VALUES[0] / 1.12, T0_PRIME_VALUES[-1] * 1.12)
    selected = [
        median_residual[network_index, :, np.flatnonzero(T0_VALUES == s)[0]]
        for s in T0_PRIME_CURVE_SIZES
    ]
    axis.set_ylim(0.0, 1.08 * np.nanmax(selected))
    axis.set_xlabel(r"$T_0'$", fontsize=8)
    axis.set_ylabel(r"$\max_n\|M^nA-I\|_{\mathrm{op}}$", fontsize=8)
    axis.set_title(NETWORK_LABELS[network], fontsize=8)
    axis.grid(True, alpha=.25)
    axis.spines[["top", "right"]].set_visible(False)
    axis.legend(frameon=False, ncol=2, fontsize=6.5, loc="upper right")
    path = OUTPUT_DIR / f"residual_vs_T0_prime_{network}.png"
    fig.savefig(path, dpi=800, bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return path


T0_PRIME_FIGURES = [
    draw_residual_vs_T0_prime(network_index, network)
    for network_index, network in enumerate(NETWORKS)
]
print("T0' 曲线:", *T0_PRIME_FIGURES, sep="\n")

## 8. 联合敏感性热图

左图给出中位残差，右图给出 20 次运行中满足 $D_M\le1/2$ 的比例。它用于确认曲线中的转折并不是单条切片造成的偶然现象。

In [ ]:

# Publication export: preserve the numerical curves and use embedded vector fonts.
plt.rcParams.update({"font.family": "DejaVu Sans", "mathtext.fontset": "stix",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7})
def draw_heatmap(network_index: int, network: str) -> Path:
    """图 C：同时显示 T0 与 T0' 的联合影响。"""
    values = median_residual[network_index]
    finite = values[np.isfinite(values) & (values > 0)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)
    image = axes[0].imshow(
        values,
        origin="lower",
        aspect="auto",
        cmap="viridis_r",
        norm=LogNorm(vmin=float(finite.min()), vmax=float(finite.max())),
    )
    axes[1].imshow(
        pass_rate[network_index],
        origin="lower",
        aspect="auto",
        cmap="YlGn",
        vmin=0.0,
        vmax=1.0,
    )
    threshold = np.sqrt(float(finite.min() * finite.max()))
    for ell in range(len(T0_PRIME_VALUES)):
        for p in range(len(T0_VALUES)):
            color = "white" if values[ell, p] > threshold else "black"
            axes[0].text(p, ell, f"{values[ell, p]:.2g}",
                         ha="center", va="center", fontsize=7, color=color)
            axes[1].text(p, ell, f"{pass_rate[network_index, ell, p]:.0%}",
                         ha="center", va="center", fontsize=7)
    axes[0].set_title(r"Median $\max_n\|M^nA-I\|_{op}$")
    axes[1].set_title(r"Fraction satisfying $D_M\leq1/2$")
    for axis in axes:
        axis.set_xticks(range(len(T0_VALUES)))
        axis.set_xticklabels([f"{value:,}" for value in T0_VALUES],
                             rotation=38, ha="right")
        axis.set_yticks(range(len(T0_PRIME_VALUES)))
        axis.set_yticklabels([str(value) for value in T0_PRIME_VALUES])
        axis.set_xlabel(r"$T_0$")
        axis.set_ylabel(r"$T_0'$")
    fig.colorbar(image, ax=axes[0], shrink=.86, label="operator-norm residual")
    fig.suptitle(NETWORK_LABELS[network], fontsize=8)
    path = OUTPUT_DIR / f"residual_heatmap_{network}.png"
    fig.savefig(path, dpi=800, bbox_inches="tight")
    fig.savefig(path.with_suffix(".pdf"), bbox_inches="tight")
    plt.close(fig)
    return path


HEATMAP_FIGURES = [
    draw_heatmap(network_index, network)
    for network_index, network in enumerate(NETWORKS)
]
print("热图:", *HEATMAP_FIGURES, sep="\n")

## 9. 自动汇总肘部位置与当前主实验参数

“elbow”采用一个可复现的几何定义：先对横轴取对数并归一化，再寻找下降曲线距离首尾连线最远的位置。它用于辅助读图，不代表理论阈值。

In [ ]:
elbow_summary = {}
for network_index, network in enumerate(NETWORKS):
    strongest_ell = int(np.flatnonzero(T0_PRIME_VALUES == T0_CURVE_ROUNDS[-1])[0])
    largest_p = int(np.flatnonzero(T0_VALUES == T0_PRIME_CURVE_SIZES[-1])[0])
    t0_knee = elbow_index(
        T0_VALUES, median_residual[network_index, strongest_ell]
    )
    rounds_knee = elbow_index(
        T0_PRIME_VALUES, median_residual[network_index, :, largest_p]
    )
    elbow_summary[network] = {
        "T0_elbow_at_T0_prime_120": int(T0_VALUES[t0_knee]),
        "T0_prime_elbow_at_T0_12800": int(T0_PRIME_VALUES[rounds_knee]),
        "residual_at_current_T0_999_T0_prime_2": float(
            median_residual[
                network_index,
                np.flatnonzero(T0_PRIME_VALUES == 2)[0],
                np.flatnonzero(T0_VALUES == 999)[0],
            ]
        ),
        "failed_fraction": float(
            np.mean(~np.isfinite(residuals[network_index]))
        ),
    }

diagnostics = {
    "runs": RUNS,
    "T0_values": T0_VALUES.tolist(),
    "T0_prime_values": T0_PRIME_VALUES.tolist(),
    "environment_seed": ENVIRONMENT_SEED,
    "trajectory_seed": TRAJECTORY_SEED,
    "gamma": GAMMA,
    "elbows": elbow_summary,
    "population_floors": {
        network: {
            str(int(rounds)): float(population_floors[network_index, ell])
            for ell, rounds in enumerate(T0_PRIME_VALUES)
        }
        for network_index, network in enumerate(NETWORKS)
    },
}
(OUTPUT_DIR / "diagnostics.json").write_text(
    json.dumps(diagnostics, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(elbow_summary, indent=2, ensure_ascii=False))

## 实际运行结果摘要（20 次独立轨迹）

- 当通信已经充分（$T_0'=120$）时，两种网络的中位残差都从 $T_0=50$ 时约 2.13，下降到 $T_0=400$ 时约 0.67，再下降到 $T_0=12800$ 时约 0.15；几何肘部位于 $T_0\approx400$。
- 固定 $T_0=12800$ 时，全连接网络的中位残差从 $T_0'=2$ 时的 1.08，下降到 $T_0'=8$ 时的 0.22；$T_0'\ge20$ 后基本停在约 0.15 的有限样本平台。
- 同样固定 $T_0=12800$ 时，环形网络从 1.11 下降到 $T_0'=8$ 时的 0.34、$T_0'=20$ 时的 0.19，直到约 80 轮才接近 0.15 的有限样本平台。它体现了环形网络传播更慢。
- 主收敛实验采用的 $T_0=999,T_0'=2$ 对应中位残差约 1.36，尚不满足 $D_M\le1/2$ 这个充分条件。这不等于算法必然发散，只说明该理论条件相对保守。

## 10. 图片

### 全连接网络：改变 $T_0$

![fully connected T0](preconditioner_sensitivity_result/residual_vs_T0_fully_connected.png)

### 全连接网络：改变 $T_0'$

![fully connected T0 prime](preconditioner_sensitivity_result/residual_vs_T0_prime_fully_connected.png)

### 环形网络：改变 $T_0$

![ring T0](preconditioner_sensitivity_result/residual_vs_T0_ring.png)

### 环形网络：改变 $T_0'$

![ring T0 prime](preconditioner_sensitivity_result/residual_vs_T0_prime_ring.png)

### 两种网络的联合敏感性

![fully connected heatmap](preconditioner_sensitivity_result/residual_heatmap_fully_connected.png)

![ring heatmap](preconditioner_sensitivity_result/residual_heatmap_ring.png)

## 11. 如何调参

只需要修改第 3 节参数单元：

- `RUNS`：独立轨迹数量，越大越平滑、运行越慢；
- `T0_VALUES`：横向研究的预采样长度；最大值决定每条轨迹需要生成多长；
- `T0_PRIME_VALUES`：测试的 ratio 通信轮数；
- `T0_CURVE_ROUNDS`：在“残差—$T_0$”图上显示哪些曲线；
- `T0_PRIME_CURVE_SIZES`：在“残差—$T_0'$”图上显示哪些曲线。

如果只想快速测试代码，可将 `RUNS=3`，并减少 `T0_VALUES`、`T0_PRIME_VALUES` 中的取值。正式作图再恢复当前设置。

## 12. 在本地 Jupyter 中重新绘图并直接显示

运行前面的实验单元后，再运行下面这个单元。它会根据内存中的实验结果重新生成六张图，并直接显示在 notebook 中；不需要提前上传图片文件夹。


In [ ]:

# Publication export: preserve the numerical curves and use embedded vector fonts.
plt.rcParams.update({"font.family": "DejaVu Sans", "mathtext.fontset": "stix",
                     "pdf.fonttype": 42, "ps.fonttype": 42, "font.size": 8,
                     "xtick.labelsize": 7, "ytick.labelsize": 7})
# ==================== 本地 Jupyter 直接绘图与显示 ====================
# 这个单元会重新生成六张图片，因此即使上传 notebook 时没有附带图片文件，
# 只要已经依次运行了上面的实验单元，这里仍然可以正常显示全部结果。
required_names = (
    "median_residual",
    "pass_rate",
    "population_floors",
    "draw_residual_vs_T0",
    "draw_residual_vs_T0_prime",
    "draw_heatmap",
)
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise RuntimeError(
        "请先从头运行前面的实验单元。当前缺少变量："
        + ", ".join(missing_names)
    )

# 第一组：固定 T0'，观察增加 T0 的效果。
jupyter_T0_figures = [
    draw_residual_vs_T0(network_index, network)
    for network_index, network in enumerate(NETWORKS)
]

# 第二组：固定 T0，观察增加 T0' 的效果。
jupyter_T0_prime_figures = [
    draw_residual_vs_T0_prime(network_index, network)
    for network_index, network in enumerate(NETWORKS)
]

# 第三组：同时展示 T0 和 T0' 联合作用的热图。
jupyter_heatmap_figures = [
    draw_heatmap(network_index, network)
    for network_index, network in enumerate(NETWORKS)
]

figure_groups = (
    ("固定 T0'，改变 T0", jupyter_T0_figures),
    ("固定 T0，改变 T0'", jupyter_T0_prime_figures),
    ("T0 与 T0' 联合敏感性", jupyter_heatmap_figures),
)

# 只使用 Matplotlib 显示刚刚生成的图片，不依赖额外的 IPython 显示代码。
for group_title, paths in figure_groups:
    print("\n" + "=" * 12, group_title, "=" * 12)
    for network, figure_path in zip(NETWORKS, paths):
        rendered_image = plt.imread(figure_path)
        height, width = rendered_image.shape[:2]
        display_width = 11.0
        display_height = display_width * height / width
        fig, axis = plt.subplots(
            figsize=(display_width, display_height), constrained_layout=True
        )
        axis.imshow(rendered_image)
        axis.set_title(NETWORK_LABELS[network], fontsize=8)
        axis.axis("off")
        # block=False 在本地 Jupyter 中会正常内嵌显示，也不会阻塞脚本。
        plt.show(block=False)
        plt.close(fig)

print("六张图片已经重新生成，并直接显示在 notebook 中。")


## 投稿用清晰图片：从保存数据快速导出
运行下一个单元即可从归档结果生成矢量 PDF、800 dpi PNG 和组合 TIFF，无需重新运行实验。曲线数值和现有坐标范围保持不变。


In [ ]:
from pathlib import Path
import runpy
for candidate in (Path.cwd(), Path.cwd() / "article-reproduction", Path.cwd().parent):
    if (candidate / "export_figures.py").is_file() and (candidate / "data").is_dir():
        runpy.run_path(str(candidate / "export_figures.py"), run_name="__main__")
        break
else:
    raise FileNotFoundError("Open this notebook from the article-reproduction folder, or its parent.")
